# GVC Measure 

Pulls OECD Trade-in-Value-Added (TiVa) API from each sector mapping to HS-6 Product Level in order to develop and create a GVC-intensity scoring measure.



In [1]:
# Pull BACII data from the database for 2017


import datetime
import os
import sys
import time 
import numpy as np
import pandas as pd
import sqlalchemy as sa
from sqlalchemy import create_engine
from sqlalchemy import MetaData, Table, Column, Integer, String, DateTime, Float
from sqlalchemy.sql import select, and_, or_, not_
from sqlalchemy import func
from sqlalchemy import text
from sqlalchemy import inspect
from sqlalchemy import exc  
from sqlalchemy import event
from sqlalchemy.pool import Pool
import logging

logging.basicConfig()
logging.getLogger('sqlalchemy.engine').setLevel(logging.INFO)

## Data Loader for Project Codes

In [2]:
import pandas as pd
def load_product_codes():
    # Load HS17 product codes from data folder
    file_path = 'data/BACI_HS17_V202601/product_codes_HS17_V202601.csv'
    
    try:
        df = pd.read_csv(file_path, dtype= 'str')
        # Fix leading zeros
        df['code'] = df['code'].str.zfill(6)  
        # Check all codes are exactly 6 digits
        lengths = df['code'].str.len().value_counts()
        print("Code length distribution:")
        print(lengths)
        
        
        print(f"Loaded {len(df)} product codes")
        print(df.head(20))
        return df
    
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        print("Available files:")
        import os
        data_dir = 'data'
        if os.path.exists(data_dir):
            print(os.listdir(data_dir))

if __name__ == "__main__":
    df_hs17 = load_product_codes()

Code length distribution:
code
6    5384
Name: count, dtype: int64
Loaded 5384 product codes
      code                                        description
0   010121           Horses: live, pure-bred breeding animals
1   010129  Horses: live, other than pure-bred breeding an...
2   010130                                        Asses: live
3   010190                            Mules and hinnies: live
4   010221           Cattle: live, pure-bred breeding animals
5   010229  Cattle: live, other than pure-bred breeding an...
6   010231          Buffalo: live, pure-bred breeding animals
7   010239  Buffalo: live, other than pure-bred breeding a...
8   010290  Bovine animals: live, other than cattle and bu...
9   010310            Swine: live, pure-bred breeding animals
10  010391  Swine: live, other than pure-bred breeding ani...
11  010392  Swine: live, other than pure-bred breeding ani...
12  010410                                        Sheep: live
13  010420                             

## HS Chapter, Subheadings, Descriptions

In [3]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import re

BASE    = "https://wits.worldbank.org/API/V1/wits/datasource/trn"
NS      = {"wits": "http://wits.worldbank.org"}
DATE_RE = re.compile(r"^\(([^)]*)\)\s*-?\s*")

def parse_description(raw: str) -> tuple[str, str]:
    m = DATE_RE.match(raw)
    if m:
        return m.group(1), raw[m.end():].strip()
    return "all", raw

# ── Layer 0 Chapter Prior (full HS6 universe, all 5 classes active) ───────────
# Single class per chapter range. Labels match plan §2.6 exactly.
# Class 5 now active — final goods scored, not gated out.

CLASS_NAMES = {
    1: "Raw/Primary",
    2: "Processed Material",
    3: "Component/Part",
    4: "Intermediate Assembly/Capital",
    5: "Final Good",
}

CHAPTER_PRIOR = [
    ( 1, 27, 1),   # live animals, food, raw materials, minerals, fuels
    (28, 40, 2),   # chemicals, plastics, rubber
    (41, 60, 3),   # leather, wood, paper, textiles (fabrics/yarns)
    (61, 63, 5),   # apparel and clothing accessories
    (64, 67, 5),   # footwear, headgear
    (68, 83, 3),   # stone, glass, ceramics, base metals, metal articles
    (84, 92, 4),   # machinery, electrical equipment, instruments
    (93, 93, 4),   # arms and ammunition
    (94, 96, 5),   # furniture, toys, miscellaneous manufactures
    (97, 97, 5),   # art, antiques
]

chapter_prior = pd.DataFrame([
    {
        "chapter":     str(ch).zfill(2),
        "prior_class": cls,
        "prior_label": CLASS_NAMES[cls],
    }
    for lo, hi, cls in CHAPTER_PRIOR
    for ch in range(lo, hi + 1)
])

print("Chapter prior class distribution:")
print(chapter_prior.groupby(["prior_class", "prior_label"])["chapter"]
      .count().rename("n_chapters").to_string())

# ── Source 1: GitHub CSV → chapter and heading text ───────────────────────────
print("\nFetching GitHub HS hierarchy CSV...")
gh = pd.read_csv(
    "https://raw.githubusercontent.com/datasets/harmonized-system/main/data/harmonized-system.csv",
    dtype=str
)
gh["level"] = gh["level"].str.strip()

chapter_text = (gh[gh["level"] == "2"]
    .rename(columns={"hscode": "chapter", "description": "chapter_text"})
    [["chapter", "chapter_text"]])

heading_text = (gh[gh["level"] == "4"]
    .rename(columns={"hscode": "heading", "description": "heading_text"})
    [["heading", "heading_text"]])

print(f"  Chapters: {len(chapter_text)}, Headings: {len(heading_text)}")

# ── Source 2: WITS TRAINS → HS6 subheading text ──────────────────────────────
print("Fetching WITS HS6 subheadings...")
resp = requests.get(f"{BASE}/product/all", timeout=60)
resp.raise_for_status()
root = ET.fromstring(resp.content)

rows = []
for prod in root.findall(".//wits:product", NS):
    code    = prod.get("productcode", "").strip()
    isgroup = prod.get("isgroup", "Yes")
    raw     = prod.findtext("wits:productdescription",
                            default="", namespaces=NS).strip()
    if isgroup == "No" and len(code) == 6 and code.isdigit():
        year_range, desc = parse_description(raw)
        rows.append({"subheading":      code.zfill(6),
                     "subheading_text": desc,
                     "year_range":      year_range})

df_sub = pd.DataFrame(rows)
print(f"  Subheadings: {len(df_sub):,}")

# ── Assemble Layer 0 ──────────────────────────────────────────────────────────
df_sub["chapter"] = df_sub["subheading"].str[:2]
df_sub["heading"]  = df_sub["subheading"].str[:4]

df_layer0 = (df_sub
    .merge(chapter_text,  on="chapter", how="left")
    .merge(heading_text,   on="heading",  how="left")
    .merge(chapter_prior,  on="chapter",  how="left")
    [["chapter", "chapter_text",
      "heading",  "heading_text",
      "subheading", "subheading_text",
      "year_range",
      "prior_class", "prior_label"]])

# ── Sanity checks ─────────────────────────────────────────────────────────────
print(f"\nRows:                  {len(df_layer0):,}")
print(f"Missing chapter text:  {df_layer0['chapter_text'].isna().sum()}")
print(f"Missing heading text:  {df_layer0['heading_text'].isna().sum()}")
print(f"Unassigned prior:      {df_layer0['prior_class'].isna().sum()}")
print(f"\nPrior class distribution (products):")
print(df_layer0.groupby(["prior_class","prior_label"], dropna=False)["subheading"]
      .count().rename("n_products").to_string())

# ── HS17 heading coverage check ───────────────────────────────────────────────
hs17_headings  = df_hs17["code"].str[:4].unique()
github_headings = set(heading_text["heading"].values)
missing_hd = [h for h in hs17_headings if h not in github_headings]
print(f"\nHS17 headings not in GitHub CSV: {len(missing_hd)}")
print(sorted(missing_hd))

df_layer0.head(10)

Chapter prior class distribution:
prior_class  prior_label                  
1            Raw/Primary                      27
2            Processed Material               13
3            Component/Part                   36
4            Intermediate Assembly/Capital    10
5            Final Good                       11

Fetching GitHub HS hierarchy CSV...
  Chapters: 97, Headings: 1229
Fetching WITS HS6 subheadings...
  Subheadings: 6,882

Rows:                  6,882
Missing chapter text:  0
Missing heading text:  89
Unassigned prior:      0

Prior class distribution (products):
prior_class  prior_label                  
1            Raw/Primary                      1348
2            Processed Material               1358
3            Component/Part                   2099
4            Intermediate Assembly/Capital    1497
5            Final Good                        580

HS17 headings not in GitHub CSV: 2
['8107', '8803']


,chapter,chapter_text,heading,heading_text,subheading,subheading_text,year_range,prior_class,prior_label
0,01,Animals; live,0101,"Horses, asses, mules and hinnies; live",010110,010110 -- (2002-2011) - Pure-bred breeding ani...,all,1,Raw/Primary
1,01,Animals; live,0101,"Horses, asses, mules and hinnies; live",010111,010111 -- (-2001) -- Pure-bred breeding animals,all,1,Raw/Primary
2,01,Animals; live,0101,"Horses, asses, mules and hinnies; live",010119,010119 -- (-2001) -- Other,all,1,Raw/Primary
3,01,Animals; live,0101,"Horses, asses, mules and hinnies; live",010120,"010120 -- (-2001) - Asses, mules and hinnies",all,1,Raw/Primary
4,01,Animals; live,0101,"Horses, asses, mules and hinnies; live",010121,010121 -- (2012-) -- Pure-bred breeding animals,all,1,Raw/Primary
5,01,Animals; live,0101,"Horses, asses, mules and hinnies; live",010129,010129 -- (2012-) -- Other,all,1,Raw/Primary
6,01,Animals; live,0101,"Horses, asses, mules and hinnies; live",010130,010130 -- (2012-) - Asses,all,1,Raw/Primary
7,01,Animals; live,0101,"Horses, asses, mules and hinnies; live",010190,010190 -- (2002-) - Other,all,1,Raw/Primary
8,01,Animals; live,0102,Bovine animals; live,010210,010210 -- (-2011) - Pure-bred breeding animals,all,1,Raw/Primary
9,01,Animals; live,0102,Bovine animals; live,010221,010221 -- (2012-) -- Pure-bred breeding animals,all,1,Raw/Primary


## Layer 1 — Zero-Shot NLI on HS6 Descriptions

Runs each HS6 short description through a zero-shot Natural Language Inference (NLI) model against five chain-position hypotheses (one per class). No training data is required — classification is driven entirely by the semantic relationship between the product description and the explicitly stated hypothesis.

**Output columns added to `df_layer1`:**
- `nli_prob_1` … `nli_prob_5` — normalised entailment probability per class  
- `nli_continuous` — probability-weighted average of class indices [1, 5]  
- `nli_class` — argmax predicted class (1–5)  
- `nli_label` — human-readable class name  
- `nli_uncertainty` — normalised entropy H / log(5), bounded [0, 1]

**Models:** `cross-encoder/nli-deberta-v3-small` (primary, open weights).  
`facebook/bart-large-mnli` is defined as a second model for ensemble comparison in later work.  
Both are run locally with no API dependency.

In [4]:
# ── Layer 1: Hypothesis definitions ──────────────────────────────────────────
# Each hypothesis is the natural-language statement of what it means for a
# product to belong to that chain-position class. Phrasing is directly from
# the Master Plan §4 Layer 1. Tested empirically against anchor products —
# performance is sensitive to wording, so treat these as v1 to be iterated.

HYPOTHESES = {
    1: (
        "This product is a raw or primary material that has been extracted, "
        "harvested, or produced in its natural state and has undergone little "
        "or no industrial transformation."
    ),
    2: (
        "This product is an industrial material that has been refined, "
        "processed, or transformed from a raw input into a standardised form, "
        "but is not yet a discrete component or assembled part of anything."
    ),
    3: (
        "This product is a discrete manufactured part or component that is "
        "designed to be incorporated into a larger assembly or system and "
        "cannot function independently."
    ),
    4: (
        "This product is either a complex sub-assembly incorporated into final "
        "goods during manufacturing, or a capital good used as equipment in a "
        "production process rather than consumed directly."
    ),
    5: (
        "This product is a complete, finished good intended for direct use by "
        "a household, business, or government without further industrial "
        "transformation."
    ),
}

CLASS_NAMES = {
    1: "Raw/Primary",
    2: "Processed Material",
    3: "Component/Part",
    4: "Intermediate Assembly/Capital",
    5: "Final Good",
}

print("Hypotheses defined:")
for cls, hyp in HYPOTHESES.items():
    print(f"  Class {cls} ({CLASS_NAMES[cls]}): {hyp[:60]}...")

Hypotheses defined:
  Class 1 (Raw/Primary): This product is a raw or primary material that has been extr...
  Class 2 (Processed Material): This product is an industrial material that has been refined...
  Class 3 (Component/Part): This product is a discrete manufactured part or component th...
  Class 4 (Intermediate Assembly/Capital): This product is either a complex sub-assembly incorporated i...
  Class 5 (Final Good): This product is a complete, finished good intended for direc...


In [5]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "transformers", "sentencepiece"],
    capture_output=True, text=True
)
print(result.stdout[-3000:])
print(result.stderr[-3000:])


c:\Users\dcost\Desktop\BSE\Term 3\Master Thesis\masters_thesis\.venv\Scripts\python.exe: No module named pip



In [6]:
# ── Layer 1: Load NLI model ───────────────────────────────────────────────────
# Primary model: DeBERTa-v3-small fine-tuned on MNLI + SNLI + ANLI.
# Chosen for: strong zero-shot NLI accuracy, small footprint (~180MB),
# locally runnable, open weights (Apache 2.0).
#
# Fallback / ensemble comparison model: facebook/bart-large-mnli.
# Not loaded by default — uncomment to run ensemble (slower, ~1.6GB).
#
# multi_label=True: each hypothesis is scored independently rather than
# as competing candidates. This is correct here because a description can
# plausibly entail more than one hypothesis at partial probability —
# we normalise the raw scores to a probability distribution ourselves.

from transformers import pipeline

MODEL_PRIMARY  = "cross-encoder/nli-deberta-v3-small"
# MODEL_ENSEMBLE = "facebook/bart-large-mnli"

print(f"Loading NLI model: {MODEL_PRIMARY} ...")
nli = pipeline(
    "zero-shot-classification",
    model=MODEL_PRIMARY,
    device=-1,           # CPU; set to 0 for first GPU if available
)
print("Model loaded.")

c:\Users\dcost\Desktop\BSE\Term 3\Master Thesis\masters_thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading NLI model: cross-encoder/nli-deberta-v3-small ...


c:\Users\dcost\Desktop\BSE\Term 3\Master Thesis\masters_thesis\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dcost\.cache\huggingface\hub\models--cross-encoder--nli-deberta-v3-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


: 

In [ ]:
# ── Layer 1: Scoring function ─────────────────────────────────────────────────
import math

def nli_score_product(description: str) -> dict:
    """
    Score one HS6 product description against all five chain-position hypotheses.

    Returns a dict with:
      nli_prob_1..5   normalised entailment probability per class
      nli_continuous  probability-weighted average of class indices [1, 5]
      nli_class       predicted class (int 1-5, argmax)
      nli_label       human-readable class name
      nli_uncertainty normalised entropy H / log(5), bounded [0, 1]

    Returns all-None dict for empty or null descriptions.
    """
    desc = str(description).strip() if description and str(description).strip() else None
    if desc is None:
        return {
            "nli_prob_1": None, "nli_prob_2": None, "nli_prob_3": None,
            "nli_prob_4": None, "nli_prob_5": None,
            "nli_continuous":  None,
            "nli_class":       None,
            "nli_label":       None,
            "nli_uncertainty": None,
        }

    # multi_label=True: independent entailment score per hypothesis
    result = nli(
        desc,
        candidate_labels=list(HYPOTHESES.values()),
        multi_label=True,
    )

    # Map hypothesis text back to class number
    hyp_to_class = {v: k for k, v in HYPOTHESES.items()}
    raw = {hyp_to_class[label]: score
           for label, score in zip(result["labels"], result["scores"])}

    # Normalise raw entailment scores to sum to 1
    total = sum(raw.values())
    norm  = {cls: raw[cls] / total for cls in sorted(raw)}

    # Continuous score: probability-weighted average of class indices
    continuous = sum(cls * norm[cls] for cls in norm)

    # Predicted class: argmax of normalised probabilities
    predicted = max(norm, key=norm.get)

    # Uncertainty: normalised Shannon entropy H / log(5), bounded [0, 1]
    # High uncertainty (→ 1.0) = model spreads probability evenly across classes
    # Low uncertainty (→ 0.0)  = model concentrates on one class
    entropy     = -sum(p * math.log(p + 1e-12) for p in norm.values())
    max_entropy = math.log(len(norm))
    uncertainty = entropy / max_entropy if max_entropy > 0 else 0.0

    return {
        "nli_prob_1":      round(norm[1], 6),
        "nli_prob_2":      round(norm[2], 6),
        "nli_prob_3":      round(norm[3], 6),
        "nli_prob_4":      round(norm[4], 6),
        "nli_prob_5":      round(norm[5], 6),
        "nli_continuous":  round(continuous, 4),
        "nli_class":       predicted,
        "nli_label":       CLASS_NAMES[predicted],
        "nli_uncertainty": round(uncertainty, 4),
    }

print("nli_score_product() defined.")

In [ ]:
# ── Layer 1: Anchor product smoke test ───────────────────────────────────────
# Run a small set of unambiguous anchor products before scoring the full
# HS6 universe. Every anchor must be classified correctly — any failure
# here is a hard problem with the hypothesis phrasing, not a borderline case.
# (Anchor list from Master Plan §6.). heheheheh

ANCHORS = [
    # (description, expected_class)
    ("Petroleum oils, crude",                               1),
    ("Iron ores and concentrates, non-agglomerated",        1),
    ("Cotton, not carded or combed",                        1),
    ("Copper, refined, in the form of cathodes",            2),
    ("Steel, flat-rolled, cold-rolled, width>=600mm",       2),
    ("Polyethylene having a specific gravity of <0.94",     2),
    ("Ball or roller bearings",                             3),
    ("Printed circuits",                                    3),
    ("Electric motors of an output exceeding 37.5 W",      3),
    ("Compression-ignition internal combustion engines",    4),
    ("Machine-tools for working metal by removal of material", 4),
    ("Motor cars and other motor vehicles for persons",     5),
    ("Telephone sets for cellular networks",                5),
    ("Wine of fresh grapes, in containers <=2 litres",      5),
]

print(f"Running anchor smoke test on {len(ANCHORS)} products...
")
print(f"{'Description':<55} {'Exp':>4} {'Got':>4} {'Score':>6} {'Unc':>5}  OK?")
print("-" * 85)

n_correct = 0
for desc, expected in ANCHORS:
    result = nli_score_product(desc)
    got    = result["nli_class"]
    score  = result["nli_continuous"]
    unc    = result["nli_uncertainty"]
    ok     = "✓" if got == expected else "✗"
    if got == expected:
        n_correct += 1
    print(f"{desc[:54]:<55} {expected:>4} {got:>4} {score:>6.3f} {unc:>5.3f}  {ok}")

print(f"
Anchor accuracy: {n_correct}/{len(ANCHORS)} = {n_correct/len(ANCHORS):.0%}")
print("Proceed to full scoring only if accuracy is high (>=80% recommended).")

In [ ]:
# ── Layer 1: Score full HS6 universe ─────────────────────────────────────────
# Applies nli_score_product() to every row of df_layer0 using the
# subheading_text column (HS6 short description from WITS).
#
# Runtime note: ~5,000 products × ~0.3s per product ≈ 25 min on CPU.
# Set device=0 in the pipeline call above if a GPU is available (~2–3 min).
# A tqdm progress bar is shown so you can monitor progress.
#
# The scored DataFrame is saved to CSV immediately after completion so
# results are not lost if the kernel is restarted.

import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas(desc="NLI scoring")

print(f"Scoring {len(df_layer0):,} products ...")

# Score each row and expand the result dict into columns
nli_results = df_layer0["subheading_text"].progress_apply(nli_score_product)
df_nli      = pd.DataFrame(nli_results.tolist())

# Concatenate with Layer 0 — preserve all existing columns, append NLI columns
df_layer1 = pd.concat([df_layer0.reset_index(drop=True),
                        df_nli.reset_index(drop=True)], axis=1)

print(f"
Layer 1 shape: {df_layer1.shape}")
print(f"Missing NLI scores: {df_layer1['nli_class'].isna().sum()}")
print(f"
Predicted class distribution:")
print(
    df_layer1
    .groupby(["nli_class", "nli_label"], dropna=False)["subheading"]
    .count()
    .rename("n_products")
    .to_string()
)

# Save immediately
out_path = "layer1_nli_scores.csv"
df_layer1.to_csv(out_path, index=False)
print(f"
Saved to {out_path}")

In [ ]:
# ── Layer 1: Score full HS6 universe ─────────────────────────────────────────
# Applies nli_score_product() to every row of df_layer0 using the
# subheading_text column (HS6 short description from WITS).
#
# Runtime note: ~5,000 products × ~0.3s per product ≈ 25 min on CPU.
# Set device=0 in the pipeline call above if a GPU is available (~2–3 min).
# A tqdm progress bar is shown so you can monitor progress.
#
# The scored DataFrame is saved to CSV immediately after completion so
# results are not lost if the kernel is restarted.

import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas(desc="NLI scoring")

print(f"Scoring {len(df_layer0):,} products ...")

# Score each row and expand the result dict into columns
nli_results = df_layer0["subheading_text"].progress_apply(nli_score_product)
df_nli      = pd.DataFrame(nli_results.tolist())

# Concatenate with Layer 0 — preserve all existing columns, append NLI columns
df_layer1 = pd.concat([df_layer0.reset_index(drop=True),
                        df_nli.reset_index(drop=True)], axis=1)

print(f"
Layer 1 shape: {df_layer1.shape}")
print(f"Missing NLI scores: {df_layer1['nli_class'].isna().sum()}")
print(f"
Predicted class distribution:")
print(
    df_layer1
    .groupby(["nli_class", "nli_label"], dropna=False)["subheading"]
    .count()
    .rename("n_products")
    .to_string()
)

# Save immediately
out_path = "layer1_nli_scores.csv"
df_layer1.to_csv(out_path, index=False)
print(f"
Saved to {out_path}")

In [ ]:
# ── Layer 1: Diagnostics ──────────────────────────────────────────────────────
# Examines agreement between Layer 0 chapter prior and Layer 1 NLI predictions,
# and flags high-uncertainty products for inspection.

import pandas as pd

# 1. Agreement rate between chapter prior and NLI prediction
df_layer1["prior_nli_agree"] = (
    df_layer1["prior_class"] == df_layer1["nli_class"]
)
agree_rate = df_layer1["prior_nli_agree"].mean()
print(f"Chapter prior / NLI agreement rate: {agree_rate:.1%}")
print("(Expected to be moderate — NLI adds within-chapter resolution)
")

# 2. Disagreement breakdown by chapter
disagree = df_layer1[~df_layer1["prior_nli_agree"]].copy()
print(f"Disagreements: {len(disagree):,} products ({len(disagree)/len(df_layer1):.1%})")
print("
Top 10 chapters by disagreement count:")
print(
    disagree.groupby("chapter")["subheading"]
    .count()
    .sort_values(ascending=False)
    .head(10)
    .to_string()
)

# 3. High-uncertainty products (uncertainty > 0.75 = model very unsure)
HIGH_UNC_THRESHOLD = 0.75
high_unc = df_layer1[df_layer1["nli_uncertainty"] > HIGH_UNC_THRESHOLD].copy()
print(f"
High-uncertainty products (unc > {HIGH_UNC_THRESHOLD}): {len(high_unc):,}")
print("Sample of high-uncertainty products (likely double-dipping candidates):")
print(
    high_unc[["subheading", "subheading_text", "nli_class",
               "nli_label", "nli_uncertainty"]]
    .sort_values("nli_uncertainty", ascending=False)
    .head(15)
    .to_string(index=False)
)

# 4. Continuous score distribution by predicted class
print("
Continuous score statistics by NLI class:")
print(
    df_layer1
    .groupby("nli_class")["nli_continuous"]
    .agg(["mean", "std", "min", "max"])
    .round(3)
    .to_string()
)

df_layer1.head(10)

In [ ]:
# ── Layer 1: Anchor validation — formal pass / fail ──────────────────────────
# Looks up the formal anchor products in the scored df_layer1 by HS6 subheading
# code. Reports predicted vs expected class and flags failures.
# (Anchor codes from Master Plan §6.)

ANCHOR_CODES = {
    # Class 1
    "270900": 1,  # Petroleum oils, crude
    "260111": 1,  # Iron ores, non-agglomerated
    "520100": 1,  # Cotton, not carded or combed
    "270112": 1,  # Bituminous coal
    "260600": 1,  # Aluminium ores and concentrates (bauxite)
    # Class 2
    "740811": 2,  # Refined copper wire
    "720917": 2,  # Cold-rolled steel, width>=600mm
    "390110": 2,  # Polyethylene <0.94 density
    "520511": 2,  # Cotton yarn, single, uncombed
    "760110": 2,  # Aluminium, not alloyed, unwrought
    # Class 3
    "848210": 3,  # Ball bearings
    "853400": 3,  # Printed circuits
    "850131": 3,  # DC motors >37.5W <=750W
    "848140": 3,  # Safety/relief valves
    "900190": 3,  # Lenses, prisms, optical elements
    # Class 4
    "840820": 4,  # Compression-ignition engines for vehicles
    "845710": 4,  # Machining centres for metal
    "847950": 4,  # Industrial robots
    "848310": 4,  # Transmission shafts and cranks
    "841430": 4,  # Compressors for refrigerating equipment
    # Class 5
    "870321": 5,  # Motor cars, spark-ignition <=1000cc
    "851712": 5,  # Telephones for cellular networks
    "845011": 5,  # Washing machines <=6kg
    "220421": 5,  # Wine in containers <=2 litres
    "610311": 5,  # Men's suits, of wool
    "841821": 5,  # Household refrigerators, compression-type
}

scored = df_layer1.set_index("subheading")
print(f"{'Code':<8} {'Expected':>8} {'Got':>4} {'Score':>6} {'Unc':>5}  OK?  Description")
print("-" * 100)

n_correct = 0
n_found   = 0
for code, expected in ANCHOR_CODES.items():
    if code not in scored.index:
        print(f"{code:<8} {'—':>8}  not found in scored dataset")
        continue
    row  = scored.loc[code]
    got  = row["nli_class"]
    ok   = "✓" if got == expected else "✗"
    if got == expected:
        n_correct += 1
    n_found += 1
    desc = str(row["subheading_text"])[:45]
    print(f"{code:<8} {expected:>8} {got:>4} "
          f"{row['nli_continuous']:>6.3f} {row['nli_uncertainty']:>5.3f}  {ok}   {desc}")

print(f"
Formal anchor accuracy: {n_correct}/{n_found} = "
      f"{n_correct/n_found:.0%}" if n_found else "No anchors found in dataset")
print("Target: ≥80% before proceeding to Layer 2.")